# Colab runner

Runs every model through `scripts/run_model.py`. Data and results are stored in Google Drive (`MyDrive/mat629-lob/`), and each run skips feature sets that are already done, so after a disconnect just re-run the setup cell and the same model cell.

Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU). Figures are saved to `results/.../figures/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WORK_DIR = "/content/drive/MyDrive/mat629-lob"
os.makedirs(WORK_DIR, exist_ok=True)

%cd /content
if os.path.exists("mat629-lob-prediction"):
    !git -C mat629-lob-prediction pull -q
else:
    !git clone -q https://github.com/lshangro-png/mat629-lob-prediction.git
%cd /content/mat629-lob-prediction
!pip install -q kagglehub pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 · Development run on the first ~1 day (`--subset 100000`)

In [ ]:
!python -u scripts/run_model.py --model cnn --subset 100000 --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model lstm --subset 100000 --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model transformer --subset 100000 --work-dir "$WORK_DIR"

## Step 2 · Full data (~12 days), 10-second horizon, all feature sets

In [ ]:
!python -u scripts/run_model.py --model cnn --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model lstm --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model transformer --work-dir "$WORK_DIR"

## Step 3 · Baselines and longer horizons (full data)

LightGBM on every feature set and horizon (CPU, a few minutes each), then the three sequence models on the `full` feature set at 10 / 30 / 60 / 300 s (with the hysteresis rule and saved predictions).

In [ ]:
!python -u scripts/run_model.py --model lgbm --horizons 10 30 60 300 --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model cnn --horizons 10 30 60 300 --feature-sets full --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model lstm --horizons 10 30 60 300 --feature-sets full --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/run_model.py --model transformer --horizons 10 30 60 300 --feature-sets full --work-dir "$WORK_DIR"

## Step 4 · Walk-forward evaluation (10 s horizon)

Rolling folds: 5 days train → 1 day validation → 1 day test, moved forward one day at a time (6 test days). Three seeds per fold plus a seed-averaged ensemble. PCA is refitted on each fold's training days. The last cell prints the per-day and aggregate tables.

In [ ]:
!python -u scripts/walk_forward.py --model lgbm --feature-sets full no_pca --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/walk_forward.py --model cnn --work-dir "$WORK_DIR"

In [ ]:
!python -u scripts/walk_forward.py --report --work-dir "$WORK_DIR"

## Summary

In [ ]:
import glob
import pandas as pd

for d in ["results/subset_100000", "results"]:
    files = sorted(glob.glob(f"{WORK_DIR}/{d}/h*/*_summary.csv"))
    if files:
        print(d)
        res = pd.concat(map(pd.read_csv, files))
        display(res.set_index(["horizon", "model", "feature_set"]).sort_index().round(4))